# Sindbis Soma Pipeline Workflow

This notebook walks through the Sindbis soma workflow in `iss_preprocess`.
It is intentionally Sindbis-first and ignores the mCherry path except where
the underlying code still reuses generic segmentation or stitching utilities.

Scope covered here:
- segment soma masks without trimming tile-edge labels
- build a canonical ROI-global soma atlas from raw masks (one stable global ID per soma)
- inspect atlas crops in the barcode reference frame
- find good reference tiles for soma bleedthrough / cluster-mean estimation
- tune soma reference thresholds before saving official outputs
- build soma-specific cluster means / bleedthrough reference
- basecall barcodes inside soma masks
- stitch soma calls into ROI-global coordinates
- tune and apply soma QC / post-basecalling thresholds
- run optional library and duplicate-barcode diagnostics

Implementation notes:
- per-tile soma masks are cropped from a precomputed canonical ROI atlas built
  once by `pipeline.somata.build_soma_atlases`.
- `iss_preprocess.pipeline.somata` is the Sindbis wrapper module so the workflow
  reads more cleanly.


In [ ]:
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.spatial.distance import cdist
from skimage.segmentation import find_boundaries

import iss_preprocess as iss
from iss_preprocess.io import get_mouse_path
from iss_preprocess.io.load import get_processed_path, load_ops
from iss_preprocess.pipeline.register import load_register_and_fill_tile
from iss_preprocess.pipeline.somata import (
    DEFAULT_SOMA_BARCODE_PREFIX,
    REQUIRED_SOMA_OPS_KEYS,
    apply_soma_qc_filters,
    basecall_somata_tiles,
    build_shared_soma_cluster_means,
    build_soma_atlases,
    collect_soma_trace_tile_stats,
    compute_soma_reference_from_tiles,
    compute_soma_reference_from_traces,
    enumerate_soma_candidate_tiles,
    extract_soma_trace_tiles,
    get_soma_atlas_paths,
    get_soma_output_paths,
    load_soma_atlas_tile,
    load_soma_traces_for_chambers,
    load_soma_traces_for_tiles,
    load_stitched_soma_calls,
    save_filtered_soma_calls,
    segment_soma_masks,
    setup_soma_calling_reference,
    stitch_soma_calls,
    summarize_soma_qc_threshold_grid,
    summarize_soma_reference_tiles,
    summarize_soma_reference_tiles_across_chambers,
    validate_soma_workflow_config,
)
from iss_preprocess.diagnostics.diag_somata import (
    plot_shared_soma_reference_diagnostics,
    summarize_cluster_sizes,
)
from iss_preprocess.vis.vis import plot_clusters

# autorefresh modules
%load_ext autoreload
%autoreload 2


## 1. Configure Dataset Paths

`data_paths` can contain one chamber or several chambers from the same mouse.
The workflow below runs per chamber.

Important:
- `setup_soma_calling_reference(...)` writes cluster means into each chamber's
  own `processed` folder
- if you want to reuse one chamber's cluster means across chambers, do that
  deliberately rather than assuming it happens automatically

In [ ]:
mouse_path = "essenbd_projdev/BRAC11398.3c"
chambers = ["01", "02"]
data_paths = [f"{mouse_path}/chamber_{ch}" for ch in chambers]

barcode_prefix = "barcode_round"
barcode_round_to_show = 2

# If None, the first barcode_soma_reference_tiles entry is used for each chamber.
example_tile = None

# Trace-cache extraction controls. Run the submit cell once after rebuilding the atlas
# or changing ops/atlas metadata; force=False makes existing valid caches reusable.
submit_soma_trace_extraction = True
force_soma_trace_extraction = False
trace_extraction_rois = None
collect_soma_trace_stats_after_extraction = True

# --- Shared-bleedthrough mode (default) ----------------------------------
# When True, build ONE soma cluster-means / bleedthrough matrix at the mouse
# level (`processed/{project}/{mouse}/somata_barcode_cluster_means.npy`) from
# reference tiles pooled across `shared_soma_chambers`. Each consuming chamber
# must have `use_shared_soma_cluster_means: true` in its ops for soma
# basecalling to read the shared file.
#
# Threshold ownership rule (strict): exactly ONE chamber in
# `shared_soma_chambers` may declare `soma_std_threshold_clustering` and
# `somata_cluster_score_thresh` in its ops. Multiple declarations or zero
# declarations raise. Bigger chamber naturally contributes more reference
# tiles; the build does not rebalance per chamber.
use_shared_soma_reference = True
shared_soma_chambers = data_paths

reference_tile_search_rois = None
reference_tile_search_limit = None
n_reference_tiles_to_use = 50

# Reference tiles can be chosen three ways:
# 1. set selected_reference_tiles explicitly (per-chamber list, shared mode
#    expects {chamber: [tiles, ...]})
# 2. set use_ops_reference_tiles = True to use ops["barcode_soma_reference_tiles"]
# 3. leave both unset/False to rank tiles from cached trace stats
selected_reference_tiles = None
use_ops_reference_tiles = False

std_threshold_candidates = [0.8, 1.0, 1.2, 1.5]
cluster_score_threshold_candidates = [0.8, 0.9, 0.95]

# Per-chamber mode (only used when use_shared_soma_reference is False): picks
# the single chamber whose stats and traces drive reference setup.
per_chamber_data_path = data_paths[0]
run_setup_reference_for_each_chamber = True
rerun_official_reference_setup = True

# Optional: path to the Sindbis library .mat file for appendix diagnostics.
sindbis_library_mat = None

data_paths


## 2. Validate The Sindbis Soma Config

The current Sindbis soma path depends on several ops keys that are not all
documented in `default_ops.yml`, so this check is worth doing up front.

In [ ]:
config_rows = []
example_tiles = {}

for data_path in data_paths:
    print("Validating config for", data_path)
    cfg = validate_soma_workflow_config(data_path)
    config_rows.append({"data_path": data_path, **cfg})
    ops = load_ops(data_path)
    example_tiles[data_path] = tuple(example_tile or ops["barcode_soma_reference_tiles"][0])

config_df = pd.DataFrame(config_rows)
display(config_df)

print("Required soma ops keys:")
print(", ".join(REQUIRED_SOMA_OPS_KEYS))
print()
print("Example tiles:")
display(pd.Series(example_tiles, name="example_tile"))

## 3. Helper Functions

These are lightweight notebook helpers for QC and visualization. They do not
change the underlying pipeline logic.

In [ ]:
from iss_preprocess.pipeline.register import load_and_register_sequencing_tile


def barcode_tile_rgb(stack, ops, round_index=0, quantile=99.9, vmin=0.05):
    stack = stack[:, :, np.argsort(ops["camera_order"]), :]
    colors = [
        [1.0, 0.0, 1.0],
        [0.0, 1.0, 1.0],
        [0.0, 1.0, 0.0],
        [1.0, 0.0, 0.0],
    ]
    vmax = [
        max(np.percentile(stack[:, :, ch, round_index], quantile), 1e-6)
        for ch in range(stack.shape[2])
    ]
    return iss.vis.vis.to_rgb(stack[:, :, :, round_index], colors=colors, vmax=vmax, vmin=vmin)


def _load_tile_rgb(data_path, tile_coors, round_to_show, quantile=99.9, vmin=0.05):
    ops = load_ops(data_path)
    cfg = validate_soma_workflow_config(data_path)
    stack, _ = load_and_register_sequencing_tile(
        data_path,
        tile_coors,
        prefix=barcode_prefix,
        suffix="max",
        filter_r=None,
        correct_channels=False,
        corrected_shifts="best",
        correct_illumination=True,
        nrounds=cfg["barcode_rounds"],
        specific_rounds=[round_to_show],
        bad_pixels_per_round=False,
    )
    return barcode_tile_rgb(stack, ops, round_index=0, quantile=quantile, vmin=vmin)


def plot_example_tile(data_path, tile_coors=None, quantile=99.9, vmin=0.05):
    """Overlay atlas-cropped soma boundaries on the barcode reference tile."""
    ops = load_ops(data_path)
    if tile_coors is None:
        tile_coors = tuple(ops["barcode_soma_reference_tiles"][0])
    cfg = validate_soma_workflow_config(data_path)
    masks = load_soma_atlas_tile(
        data_path,
        tile_coors=tile_coors,
        expected_reference_prefix=cfg["reference_prefix"],
        expected_corrected_shifts=cfg["corrected_shifts"],
    )
    rgb = _load_tile_rgb(
        data_path, tile_coors, barcode_round_to_show, quantile=quantile, vmin=vmin
    )
    boundaries = find_boundaries(masks)

    plt.figure(figsize=(10, 10))
    plt.imshow(rgb)
    plt.imshow(boundaries, cmap="Reds", alpha=boundaries.astype(float))
    plt.title(f"{data_path} | tile {tile_coors} | barcode round {barcode_round_to_show}")
    plt.axis("off")
    plt.show()


def plot_filtered_tile(
    data_path,
    tile_coors,
    survivors_df,
    quantile=99.9,
    vmin=0.05,
):
    """Per-tile QC overlay: barcode round + atlas mask boundaries + survivor dots."""
    cfg = validate_soma_workflow_config(data_path)
    masks = load_soma_atlas_tile(
        data_path,
        tile_coors=tile_coors,
        expected_reference_prefix=cfg["reference_prefix"],
        expected_corrected_shifts=cfg["corrected_shifts"],
    )
    rgb = _load_tile_rgb(
        data_path, tile_coors, barcode_round_to_show, quantile=quantile, vmin=vmin
    )
    boundaries = find_boundaries(masks)

    plt.figure(figsize=(10, 10))
    plt.imshow(rgb)
    plt.imshow(boundaries, cmap="Reds", alpha=boundaries.astype(float))
    if len(survivors_df):
        plt.scatter(
            survivors_df["x_in_tile"],
            survivors_df["y_in_tile"],
            facecolors="none",
            edgecolors="orange",
            s=20,
            linewidths=2,
        )
    plt.title(
        f"{data_path} | tile {tile_coors} | round {barcode_round_to_show} | "
        f"{len(survivors_df)} survivors"
    )
    plt.axis("off")
    plt.show()


def plot_filtered_roi_overview(
    data_path,
    roi,
    downsample_factor=4,
    quantile=99.9,
    vmin=0.05,
):
    """Render one ROI overlay inline (delegates to diag_somata).

    Returns the saved PNG path under
    ``processed/{chamber}/figures/filtered_soma_overlays/``.
    """
    from iss_preprocess.diagnostics.diag_somata import (
        plot_filtered_soma_overlay_roi,
    )

    return plot_filtered_soma_overlay_roi(
        data_path,
        roi=int(roi),
        barcode_prefix=barcode_prefix,
        downsample_factor=downsample_factor,
        quantile=quantile,
        vmin=vmin,
        use_slurm=False,
        show=True,
    )


def plot_qc_histograms(df, title=None):
    cols = ["dot_product_score", "mean_intensity", "mean_score", "area", "std"]
    fig, axes = plt.subplots(1, len(cols), figsize=(4 * len(cols), 3))
    for ax, col in zip(axes, cols):
        ax.hist(df[col], bins=100)
        ax.set_title(col)
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def plot_roi_positions(df, highlight_sequence=None, clip_counts=10):
    if df.empty:
        print("No rows to plot.")
        return

    plot_df = df.copy()
    value_counts = plot_df["bases"].value_counts()
    plot_df["barcode_count"] = plot_df["bases"].map(value_counts)

    for roi in sorted(plot_df["roi"].unique()):
        roi_points = plot_df[plot_df["roi"] == roi]
        plt.figure(figsize=(10, 10))
        plt.scatter(
            roi_points["x"],
            roi_points["y"],
            c=np.clip(roi_points["barcode_count"], 0, clip_counts),
            cmap="viridis",
            vmin=0,
            vmax=clip_counts,
            s=10,
        )
        if highlight_sequence is not None:
            highlight = roi_points[roi_points["bases"] == highlight_sequence]
            if len(highlight):
                plt.scatter(highlight["x"], highlight["y"], c="red", s=10)
        plt.colorbar(label="barcode_count")
        plt.title(f"ROI {roi}")
        plt.axis("off")
        plt.show()


def load_sindbis_library_sequences(mat_path):
    import scipy.io

    library = scipy.io.loadmat(mat_path)
    seqs = library["refbarcodes"][:, :17].tolist()
    seqs = ["".join(map(chr, seq)) for seq in seqs]
    return [seq[:8] + seq[10:] for seq in seqs]


def encode_sequences(sequences):
    mapping = {"A": 0, "T": 1, "G": 2, "C": 3, "N": 4}
    arr = np.array([list(seq) for seq in sequences])
    return np.vectorize(mapping.get)(arr)


def min_hamming_distance_to_library(query_sequences, library_sequences):
    if len(query_sequences) == 0 or len(library_sequences) == 0:
        return np.array([])
    query_enc = encode_sequences(query_sequences)
    library_enc = encode_sequences(library_sequences)
    return np.min(cdist(query_enc, library_enc, metric="hamming"), axis=1) * query_enc.shape[1]


## 3b. Tune Cellpose Hyperparameters On One Tile Per Chamber

Iterate on cellpose hyperparameters on a single example tile per chamber
before committing to a chamber-wide segmentation. The preview helper:

- loads the segmentation-acquisition tile the same way the segmenter does
  (single-channel path for `cellpose_channels=[c]`, registered multi-channel
  otherwise),
- runs `cellpose_segmentation` with ops defaults merged with any kwargs you
  pass (e.g. `diameter=`, `flow_threshold=`, `cellprob_threshold=`,
  `min_pix=`, `dilate_pix=`, `rescale=`, `model_type=`, `normalize=...`),
- renders one panel per channel with mask boundaries overlaid,
- returns `(masks, fig, effective_params)` — no files written.

Tweak the override kwargs, re-run, and once you're happy update the chamber's
`ops.yml` to match before running section 4.


In [ ]:
from iss_preprocess.diagnostics.diag_somata import (
    preview_cellpose_segmentation_on_tile,
)

# Override any of: diameter, flow_threshold, cellprob_threshold, min_pix,
# dilate_pix, rescale, model_type, pretrained_model, normalize. Unknown
# kwargs are forwarded to CellposeModel.eval. Leave the dict empty to use
# the chamber's current ops values.
cellpose_overrides = {
    "diameter": 40,
    "flow_threshold": 1.3,
    "cellprob_threshold": -4.5,
}

cellpose_preview = {}
for data_path in data_paths:
    masks, fig, effective_params = preview_cellpose_segmentation_on_tile(
        data_path,
        tile_coors=example_tiles[data_path],
        use_raw_stack=False,
        **cellpose_overrides,
    )
    cellpose_preview[data_path] = {
        "masks": masks,
        "params": effective_params,
    }

pd.DataFrame(
    {dp: r["params"] for dp, r in cellpose_preview.items()}
).T


## 4. Segment Soma Masks

This runs Cellpose on the acquisition specified by `ops["segmentation_acquisition"]`.
The Sindbis notebooks were using the 2D path (`use_raw_stack=False`).

In [ ]:
for data_path in data_paths:
    segment_soma_masks(
        data_path,
        use_gpu=False,
        rerun_cellpose=True,
        use_slurm=True,
        use_raw_stack=False,
    )

## 4b. Inspect Cellpose Segmentation On Reference Tiles

Quick visual QC of cellpose output before building the atlas. Renders the
segmentation-acquisition image (NeuN by default, plus DAPI if cellpose was
fed both channels) with the saved per-tile mask boundaries overlaid.

Tiles default to `ops["barcode_soma_reference_tiles"]`; cap with `n_tiles` or
pass an explicit `tiles=[...]` to override.


In [ ]:
from iss_preprocess.diagnostics.diag_somata import (
    plot_cellpose_segmentation_reference_tiles,
)

n_segmentation_qc_tiles = 4  # set to None to plot every reference tile

for data_path in data_paths:
    plot_cellpose_segmentation_reference_tiles(
        data_path,
        n_tiles=n_segmentation_qc_tiles,
        use_raw_stack=False,
    )

## 5. Obtain transform to reference acquisition
check if the segmentation round has been registered to the sequencing reference round. If not launch register to reference, then correct shifts to reg as dependency

run 
'register_to_reference'
and
correct_ref_shifts

from cli

## 6. Build Canonical ROI-Global Soma Atlas

Build one canonical soma-mask atlas per ROI in the barcode-reference frame.
Each soma is assigned a stable global integer ID; per-tile masks consumed by
`basecall_somata_tile` are deterministic crops of this atlas.

This step reads raw `<segmentation_prefix>_masks_*.npy` tiles by default
(`mask_suffix=""`), composes one direct mask-to-global-reference affine per
tile, expands the atlas canvas before warping, and merges tile-border duplicates
by containment. It must use the same `reference_prefix` and `corrected_shifts`
that basecalling will use for `load_register_and_fill_tile(...)`.

Outputs per ROI under `processed/cells/`:
- `soma_atlas_{roi}.npy` — uint32 label image
- `soma_atlas_{roi}_meta.npz` — tile origins, canvas offset, owner table,
  source provenance, and frame metadata


In [ ]:
for data_path in data_paths:
    cfg = validate_soma_workflow_config(data_path)
    slurm_folder = Path.home() / "slurm_logs" / data_path / "soma_atlas"
    slurm_folder.mkdir(parents=True, exist_ok=True)
    build_soma_atlases(
        data_path,
        reference_prefix=cfg["reference_prefix"],
        corrected_shifts=cfg["corrected_shifts"],
        use_slurm=True,
        slurm_folder=slurm_folder,
    )

## 7. Inspect Example Tiles In The Barcode Reference Frame

`load_soma_atlas_tile(...)` slices the canonical atlas built in section 6
into per-barcode-tile mask images. Each soma carries its stable global ID as
its integer label, and appears in exactly one tile. The atlas owner rule uses
tiles containing gid pixels, then picks closest tile centre to soma centroid,
then larger pixel coverage, then lexicographic tile index.


In [ ]:
for data_path in data_paths:
    plot_example_tile(data_path, tile_coors=example_tiles[data_path], quantile=99.999, vmin=0.01)

## 8. Extract Soma Traces Once

Submit one trace-extraction job per tile after the atlas has been built. Each
job writes a validated per-tile trace cache and per-tile stats under
`processed/somata_traces/`.

This is the expensive image-loading step. Downstream reference ranking,
bleedthrough/reference generation, and soma basecalling read these cached
traces and fail fast if a cache is missing or stale.


In [ ]:
trace_extraction_jobs = {}

if submit_soma_trace_extraction:
    for data_path in data_paths:
        job_ids, failed_job = extract_soma_trace_tiles(
            data_path,
            use_rois=trace_extraction_rois,
            use_slurm=True,
            force=force_soma_trace_extraction,
        )
        trace_extraction_jobs[data_path] = {
            "job_ids": job_ids,
            "failed_job": failed_job,
        }
else:
    print("Skipping trace extraction submission; existing validated caches will be used.")


In [ ]:
# Run this after the trace-extraction jobs have completed.
soma_trace_tile_stats = {}

if collect_soma_trace_stats_after_extraction:
    for data_path in data_paths:
        stats = collect_soma_trace_tile_stats(data_path)
        soma_trace_tile_stats[data_path] = stats
        print(f"Collected cached trace stats for {len(stats)} tiles in {data_path}")

    for data_path in data_paths:
        print(data_path)
        display(soma_trace_tile_stats[data_path].head(10))
else:
    print("Skipping stats collection; reference ranking will load the saved stats file if needed.")


## 9. Tune Bleedthrough And Reference-Tile Parameters

This section is the main parameter-selection module for the notebook.

In **shared mode** (`use_shared_soma_reference = True`, the default), tile
ranking, trace pooling, threshold sweep, and the diagnostic clustering preview
all operate **across `shared_soma_chambers`**, so the chosen thresholds and
the resulting bleedthrough matrix are mouse-wide. The bigger chamber
naturally contributes more reference tiles; the build does not rebalance
per chamber.

In **per-chamber mode** (`use_shared_soma_reference = False`), the section
operates only on `per_chamber_data_path`.

Use it to:
- rank candidate reference tiles from cached trace stats
- preview top tiles in the barcode reference frame
- choose `barcode_soma_reference_tiles`
- choose `soma_std_threshold_clustering`
- choose `somata_cluster_score_thresh`

If you inspected overview images elsewhere, set `selected_reference_tiles` or
`use_ops_reference_tiles = True` and skip cached-stat ranking in this section.


In [ ]:
def _resolve_pool_threshold(chambers, ops_key):
    """Return the unique declared value of `ops_key` across the chambers."""
    declared = [
        (dp, load_ops(dp).get(ops_key))
        for dp in chambers
        if load_ops(dp).get(ops_key) is not None
    ]
    if len(declared) != 1:
        # check if all declared values are the same and warn if so, but still return the value
        different_values = len(set(val for _, val in declared))
        if different_values == 1:
            print(
                f"Warning: Multiple chambers declare {ops_key} but with the same value "
                f"{declared[0][1]}. This is allowed."
            )
        else:
            raise ValueError(
                f"Expected exactly one key value declaration for {ops_key} across chambers, but found {len(declared)}: "
                + ", ".join(f"{dp}={val}" for dp, val in declared)
            )
    return declared[0][1]


if use_shared_soma_reference:
    # Mouse-wide ranking across `shared_soma_chambers`.
    pool_std_threshold = _resolve_pool_threshold(
        shared_soma_chambers, "soma_std_threshold_clustering"
    )

    if selected_reference_tiles is not None:
        # Caller-supplied {chamber: [tiles]} mapping wins.
        if not isinstance(selected_reference_tiles, dict):
            raise TypeError(
                "In shared mode, set selected_reference_tiles to a "
                "{chamber_data_path: [tiles, ...]} mapping."
            )
        tuned_reference_tiles_by_chamber = {
            dp: [tuple(t) for t in tiles]
            for dp, tiles in selected_reference_tiles.items()
        }
        reference_tile_summary = pd.DataFrame()
        print("Using selected_reference_tiles from the notebook config.")
    elif use_ops_reference_tiles:
        tuned_reference_tiles_by_chamber = {}
        for dp in shared_soma_chambers:
            tuned_reference_tiles_by_chamber[dp] = [
                tuple(t)
                for t in load_ops(dp).get("barcode_soma_reference_tiles", [])
            ]
        reference_tile_summary = pd.DataFrame()
        print("Using barcode_soma_reference_tiles from each chamber's ops.")
    else:
        reference_tile_summary = summarize_soma_reference_tiles_across_chambers(
            shared_soma_chambers,
            std_threshold=pool_std_threshold,
            use_rois=reference_tile_search_rois,
        )
        if reference_tile_search_limit is not None:
            reference_tile_summary = reference_tile_summary.head(
                reference_tile_search_limit
            )
        display(reference_tile_summary.head(20))

        top_n = reference_tile_summary.head(n_reference_tiles_to_use)
        tuned_reference_tiles_by_chamber = {dp: [] for dp in shared_soma_chambers}
        for _, row in top_n.iterrows():
            tuned_reference_tiles_by_chamber[row["chamber_data_path"]].append(
                (int(row["roi"]), int(row["tilex"]), int(row["tiley"]))
            )
        per_chamber_counts = top_n.groupby("chamber").size().to_dict()
        print(
            f"Scored {len(reference_tile_summary)} candidate tiles across "
            f"{len(shared_soma_chambers)} chambers. Top-{n_reference_tiles_to_use} "
            f"contributions per chamber: {per_chamber_counts}"
        )

    tuned_reference_tiles = [
        tile
        for tiles in tuned_reference_tiles_by_chamber.values()
        for tile in tiles
    ]

    if not tuned_reference_tiles:
        raise ValueError(
            "No reference tiles selected across the shared pool. Set "
            "selected_reference_tiles, enable use_ops_reference_tiles, or "
            "collect trace stats and rank from cache."
        )

    print("Reference tiles selected per chamber:")
    for dp, tiles in tuned_reference_tiles_by_chamber.items():
        print(f"  {dp}: {len(tiles)} tiles")
else:
    # Single-chamber mode.
    ops_reference_tiles = [
        tuple(tile)
        for tile in load_ops(per_chamber_data_path).get(
            "barcode_soma_reference_tiles", []
        )
    ]

    if selected_reference_tiles is not None:
        reference_tile_summary = pd.DataFrame()
        tuned_reference_tiles = [tuple(tile) for tile in selected_reference_tiles]
        print("Using selected_reference_tiles from the notebook config.")
    elif use_ops_reference_tiles:
        reference_tile_summary = pd.DataFrame()
        tuned_reference_tiles = ops_reference_tiles
        print("Using barcode_soma_reference_tiles from ops.")
    else:
        candidate_tiles = enumerate_soma_candidate_tiles(
            per_chamber_data_path,
            use_rois=reference_tile_search_rois,
        )
        if reference_tile_search_limit is not None:
            candidate_tiles = candidate_tiles[:reference_tile_search_limit]

        reference_tile_summary = summarize_soma_reference_tiles(
            per_chamber_data_path,
            tile_list=candidate_tiles,
        )
        display(reference_tile_summary.head(20))

        tuned_reference_tiles = [
            tuple(x)
            for x in reference_tile_summary[["roi", "tilex", "tiley"]]
            .head(n_reference_tiles_to_use)
            .to_numpy()
        ]
        print(
            f"Scored {len(reference_tile_summary)} candidate tiles for "
            f"{per_chamber_data_path}"
        )

    tuned_reference_tiles_by_chamber = {
        per_chamber_data_path: tuned_reference_tiles
    }

    if not tuned_reference_tiles:
        raise ValueError(
            "No reference tiles selected. Set selected_reference_tiles, enable "
            "use_ops_reference_tiles, or collect trace stats and rank from cache."
        )

    print("Reference tiles selected for tuning/reference setup:")
    print(tuned_reference_tiles)
    print("Current ops reference tiles:")
    print(ops_reference_tiles)


In [ ]:
# Inspect the per-chamber tile pool that the build will consume.
tuned_reference_tiles_by_chamber


In [ ]:
tuned_reference_tiles


In [ ]:
for chamber_dp, tiles in tuned_reference_tiles_by_chamber.items():
    for tile_coors in tiles:
        plot_example_tile(chamber_dp, tile_coors=tile_coors)


In [ ]:
if use_shared_soma_reference:
    candidate_reference_traces = load_soma_traces_for_chambers(
        tuned_reference_tiles_by_chamber,
    )
    initial_cluster_mean_for_preview = load_ops(shared_soma_chambers[0])[
        "initial_cluster_means"
    ]
else:
    candidate_reference_traces = load_soma_traces_for_tiles(
        per_chamber_data_path,
        tile_list=tuned_reference_tiles,
    )
    initial_cluster_mean_for_preview = load_ops(per_chamber_data_path)[
        "initial_cluster_means"
    ]

print(f"Loaded {len(candidate_reference_traces)} cached candidate soma traces")
candidate_reference_traces.head()


In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(np.log2(candidate_reference_traces["std"]), bins=60)
plt.xlabel("soma trace std (log2)")
plt.ylabel("count")
plt.title("Reference-tile soma std distribution")
plt.show()

In [ ]:
reference_parameter_rows = []

for std_threshold in std_threshold_candidates:
    for cluster_score_thresh in cluster_score_threshold_candidates:
        result = compute_soma_reference_from_traces(
            candidate_reference_traces,
            std_threshold=std_threshold,
            cluster_score_thresh=cluster_score_thresh,
            initial_cluster_mean=initial_cluster_mean_for_preview,
        )
        cluster_sizes = summarize_cluster_sizes(result["cluster_inds"])
        row = {
            "std_threshold": std_threshold,
            "cluster_score_thresh": cluster_score_thresh,
            "n_reference_somata": len(result["filtered_traces"]),
            "total_n_unassigned": int(cluster_sizes["n_unassigned"].sum()),
        }
        for col in [c for c in cluster_sizes.columns if c.startswith("cluster_")]:
            row[f"total_{col}"] = int(cluster_sizes[col].sum())
        reference_parameter_rows.append(row)

reference_parameter_summary = pd.DataFrame(reference_parameter_rows)
display(reference_parameter_summary)


In [ ]:
# `n_reference_somata` only depends on std_threshold (it's the std-filter count),
# so use total_n_unassigned to get a metric that actually reacts to
# cluster_score_thresh. Per-round cluster assignments sum across rounds, so
# divide by the number of rounds to get a per-round average count.
_nrounds_for_plot = load_ops(
    shared_soma_chambers[0] if use_shared_soma_reference else per_chamber_data_path
)["barcode_rounds"]
plot_df = reference_parameter_summary.copy()
plot_df["n_assigned_per_round"] = (
    plot_df["n_reference_somata"]
    - plot_df["total_n_unassigned"] / _nrounds_for_plot
)

fig, ax = plt.subplots(figsize=(6, 4))
for score in sorted(plot_df["cluster_score_thresh"].unique()):
    sub = plot_df[plot_df["cluster_score_thresh"] == score].sort_values("std_threshold")
    ax.plot(
        sub["std_threshold"],
        sub["n_assigned_per_round"],
        marker="o",
        label=f"cluster score > {score}",
    )
ax.set_xlabel("std threshold")
ax.set_ylabel("avg cluster-assigned somata per round")
ax.set_title("Cluster-assigned somata vs (std, cluster score) thresholds")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


In [ ]:
preview_std_threshold = std_threshold_candidates[1]
preview_cluster_score_thresh = cluster_score_threshold_candidates[-1]

tuned_reference_preview = compute_soma_reference_from_traces(
    candidate_reference_traces,
    std_threshold=preview_std_threshold,
    cluster_score_thresh=preview_cluster_score_thresh,
    initial_cluster_mean=initial_cluster_mean_for_preview,
)

print(f"Filtered reference somata: {len(tuned_reference_preview['filtered_traces'])}")

# Render all four diagnostic groups inline (no save_dir).
diag_figures = plot_shared_soma_reference_diagnostics(
    tuned_reference_preview,
    tuned_reference_tiles_by_chamber,
    save_dir=None,
)
display(diag_figures["cluster_sizes"][1])
display(diag_figures["chamber_contribution"][1])


In [ ]:
save_tuned_reference = False

if use_shared_soma_reference:
    print(
        "Shared mode: the tuned reference is built and saved by the section-10 "
        "cell below. This per-chamber save cell is a no-op in shared mode."
    )
elif save_tuned_reference:
    tuned_reference_saved = compute_soma_reference_from_tiles(
        per_chamber_data_path,
        reference_tiles=tuned_reference_tiles,
        std_threshold=preview_std_threshold,
        cluster_score_thresh=preview_cluster_score_thresh,
        save=True,
    )
    tuned_reference_saved["saved_paths"]
else:
    print(
        "Set save_tuned_reference = True to write the tuned soma reference "
        "files for this chamber (per-chamber mode only)."
    )


## 10. Build Soma-Specific Cluster Means / Bleedthrough Reference

In **shared mode** (`use_shared_soma_reference = True`, the default), one
cluster-means / bleedthrough matrix is built from the pooled reference tiles
across `shared_soma_chambers` and saved at the mouse level
(`processed/{project}/{mouse}/somata_barcode_cluster_means.npy`). Diagnostics
PNGs/CSVs are written next to it under `diagnostics/shared_soma_reference/`.

For each chamber that should consume the shared file during basecalling,
add `use_shared_soma_cluster_means: true` to its ops. Threshold ownership
is strict: exactly one chamber may declare `soma_std_threshold_clustering`
and `somata_cluster_score_thresh` in the pool.

In **per-chamber mode** (`use_shared_soma_reference = False`), the per-chamber
loop runs `setup_soma_calling_reference` per chamber, writing
`somata_barcode_cluster_means.npy` into each chamber's own `processed`
folder.

Notes:
- this step reads selected reference traces from the validated per-tile cache
- update `barcode_soma_reference_tiles`, `soma_std_threshold_clustering`, and
  `somata_cluster_score_thresh` in ops before running the per-chamber setup
- if you saved a tuned reference above, either update ops to match those
  choices or set `rerun_official_reference_setup = False` to avoid
  overwriting the tuned files


In [ ]:
build_shared_on_slurm = True

if use_shared_soma_reference:
    mouse_rel = Path(shared_soma_chambers[0]).parent
    slurm_folder = Path.home() / "slurm_logs" / mouse_rel / "shared_soma_reference"
    slurm_folder.mkdir(parents=True, exist_ok=True)
    shared_result = build_shared_soma_cluster_means(
        shared_soma_chambers,
        chamber_to_tiles=tuned_reference_tiles_by_chamber,
        std_threshold=preview_std_threshold,
        cluster_score_thresh=preview_cluster_score_thresh,
        use_slurm=build_shared_on_slurm,
        slurm_folder=slurm_folder,
        scripts_name="build_shared_soma_cluster_means",
    )
    if isinstance(shared_result, dict):
        print("Shared soma cluster means saved to:")
        for key, path in (shared_result.get("saved_paths") or {}).items():
            print(f"  {key}: {path}")
        print(f"Diagnostics dir: {shared_result.get('diagnostics_dir')}")
    else:
        # Slurm submission: result is the job id; resolve expected paths from
        # the same helper the build will use on the worker.
        outputs = get_soma_output_paths(
            shared_soma_chambers[0], barcode_prefix=barcode_prefix
        )
        print(f"Submitted slurm job {shared_result}. Expected outputs:")
        for key in [
            "shared_somata_cluster_means",
            "shared_somata_reference_barcodes",
            "shared_somata_traces",
            "shared_somata_diagnostics_dir",
        ]:
            print(f"  {key}: {outputs[key]}")
    print()
    print(
        "Set `use_shared_soma_cluster_means: true` in each consuming chamber's "
        "ops before running soma basecalling."
    )
else:
    reference_paths = (
        data_paths if run_setup_reference_for_each_chamber else [data_paths[0]]
    )
    if rerun_official_reference_setup:
        for data_path in reference_paths:
            slurm_folder = Path.home() / "slurm_logs" / data_path / "somata_reference"
            slurm_folder.mkdir(parents=True, exist_ok=True)
            setup_soma_calling_reference(
                data_path,
                use_slurm=True,
                slurm_folder=slurm_folder,
                scripts_name="setup_soma_calling_reference",
            )
    else:
        print(
            "Skipping official soma reference setup to avoid overwriting "
            "tuned reference outputs."
        )
        print(
            "Re-enable this after updating ops with the chosen reference "
            "tiles and thresholds."
        )


In [ ]:
if use_shared_soma_reference:
    outputs = get_soma_output_paths(shared_soma_chambers[0], barcode_prefix=barcode_prefix)
    print("Shared (mouse-level) soma outputs:")
    for key in [
        "shared_somata_cluster_means",
        "shared_somata_reference_barcodes",
        "shared_somata_traces",
        "shared_somata_diagnostics_dir",
    ]:
        print(f"  {key}: {outputs[key]}")
    print()
    for data_path in shared_soma_chambers:
        outputs = get_soma_output_paths(data_path, barcode_prefix=barcode_prefix)
        print(data_path)
        for key in ["somata_trace_cache_dir", "somata_trace_tile_stats"]:
            print(f"  {key}: {outputs[key]}")
        print()
else:
    reference_paths = (
        data_paths if run_setup_reference_for_each_chamber else [data_paths[0]]
    )
    for data_path in reference_paths:
        print(data_path)
        outputs = get_soma_output_paths(data_path, barcode_prefix=barcode_prefix)
        for key in [
            "somata_trace_cache_dir",
            "somata_trace_tile_stats",
            "somata_traces",
            "somata_cluster_means",
            "somata_reference_barcodes",
        ]:
            print(f"  {key}: {outputs[key]}")
        print()


## 11. Basecall Somata On All Tiles

This submits the tile-wise `basecall_somata_tile` jobs. The official path loads
validated per-tile soma trace caches, applies the saved soma cluster means, and
writes the same per-tile soma-call output files as before.

Each output dataframe still contains soma centroids in the local barcode
reference-tile frame.


In [ ]:
basecall_jobs = {}

for data_path in data_paths:
    job_ids, failed_job = basecall_somata_tiles(data_path)
    basecall_jobs[data_path] = {"job_ids": job_ids, "failed_job": failed_job}

basecall_jobs

## 12. Stitch Soma Calls Into ROI-Global Coordinates

Converts local reference-tile coordinates into ROI-global reference coords.

With the canonical atlas, each `label` is already a stable global soma ID
and owner-tile assignment guarantees per-tile uniqueness, so the stitching
dedup is a trivial `drop_duplicates(subset='label')`. The historical
tile-center distance heuristic is no longer used.


In [ ]:
stitch_jobs = {}

for data_path in data_paths:
    slurm_folder = Path.home() / "slurm_logs" / data_path / "somata_stitching"
    slurm_folder.mkdir(parents=True, exist_ok=True)
    stitch_jobs[data_path] = stitch_soma_calls(
        data_path,
        use_slurm=True,
        slurm_folder=slurm_folder,
        scripts_name="stitch_soma_calls",
    )

stitch_jobs

## 13. Tune Post-Basecalling QC Thresholds

This is the second parameter-selection module in the notebook.

Use it to sweep the soma QC thresholds on a stitched chamber table before
deciding on the final filtering rule. This is the Sindbis soma equivalent of
basecalling-parameter tuning for spot-based workflows.

In [ ]:
qc_tuning_data_path = data_paths[0]
stitched_for_qc_tuning = load_stitched_soma_calls(
    qc_tuning_data_path,
    barcode_prefix=barcode_prefix,
    filtered=False,
)

qc_threshold_grid = {
    "mean_intensity": [0.6, 0.7, 0.8],
    "std": [0.7, 0.9, 1.1],
    "dot_product_score": [0.05, 0.08, 0.1],
    "mean_score": [0.75, 0.85, 0.9],
    "area": [300, 500, 700],
}

qc_grid_summary = summarize_soma_qc_threshold_grid(
    stitched_for_qc_tuning,
    qc_threshold_grid,
)
display(qc_grid_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(
    stitched_for_qc_tuning['dot_product_score'],
    stitched_for_qc_tuning['mean_intensity'],
    c=stitched_for_qc_tuning['std'],
    cmap='viridis',
    s=2,
    alpha=0.3,
)
axes[0].set_xlabel('dot_product_score')
axes[0].set_ylabel('mean_intensity')
axes[0].set_title('All stitched somata')

axes[1].scatter(
    qc_grid_summary['n_somata'],
    qc_grid_summary['duplicate_row_fraction'],
    s=12,
    alpha=0.6,
)
axes[1].set_xlabel('somata kept')
axes[1].set_ylabel('duplicate row fraction')
axes[1].set_title('QC threshold grid summary')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 14. Define And Apply Final QC Thresholds

Adjust these thresholds per chamber. The values below mirror the rough ranges
used in the exploratory notebooks and are only starting points.

> **TODO — soma QC migration**: this section (and `save_filtered_soma_calls`
> below) duplicates work that the downstream QC pipeline does in
> `iss-qc-sindbis`. Plan is to remove `apply_soma_qc_filters` /
> `save_filtered_soma_calls` from this preprocess notebook once the
> `qc_review_v2.ipynb` ingest is established as the single source of truth
> for soma filtering. Leaving both in place for now; the QC pipeline reads
> the *unfiltered* stitched table via
> `register_somata_to_global_volume(..., filtered=False)`.


In [ ]:
qc_thresholds = {}

for data_path in data_paths:
    qc_thresholds[data_path] = {
        "mean_intensity": 0.85,
        "std": 0.8,
        "dot_product_score": 0.08,
        "mean_score": 0.85,
        "area": 400,
    }
qc_thresholds

In [ ]:
filtered_tables = {}

for data_path in data_paths:
    stitched_df = load_stitched_soma_calls(data_path, barcode_prefix=barcode_prefix, filtered=False)
    filtered_df = apply_soma_qc_filters(stitched_df, qc_thresholds[data_path])
    save_path = save_filtered_soma_calls(data_path, filtered_df, barcode_prefix=barcode_prefix)
    filtered_tables[data_path] = filtered_df
    print(f"{data_path}: kept {len(filtered_df)} / {len(stitched_df)} somata -> {save_path}")

## 14b. Spatial QC: Filtered-Soma Overlays

Two visual sanity checks on the filtered output:

- **Per-tile**: for every tile containing surviving somata, plot the barcode
  reference round (round `barcode_round_to_show`) as RGB background, overlay
  the atlas mask boundaries, and dot each surviving soma centroid (local
  `x_in_tile`/`y_in_tile`).
- **Per-ROI**: for every ROI containing surviving somata, stitch the
  reference-prefix round into an overview image and dot the surviving soma
  centroids in global `x`/`y` coords. No masks at this scale.

Both are gated by `qc_overlay_chambers` and per-figure caps
(`max_qc_tiles_to_plot`, `max_qc_rois_to_plot`) so the cell runs in a
reasonable time. Increase the caps or set to `None` to plot everything.


In [ ]:
# Per-tile overlay: fan out one slurm job per tile that contains surviving
# somata. Each job renders the barcode-round RGB + atlas mask boundaries +
# orange survivor dots in local (x_in_tile, y_in_tile) coords. Tiles without
# survivors are not plotted (unlike the per-ROI version, which renders all
# ROIs for uniformity).
from iss_preprocess.diagnostics.diag_somata import plot_filtered_soma_overlay_tiles

qc_overlay_chambers = data_paths

filtered_overlay_tile_jobs = {}
for data_path in qc_overlay_chambers:
    filtered_overlay_tile_jobs[data_path] = plot_filtered_soma_overlay_tiles(
        data_path,
        barcode_prefix=barcode_prefix,
        barcode_round_to_show=barcode_round_to_show,
        use_slurm=True,
    )
    print(
        f"{data_path}: submitted {len(filtered_overlay_tile_jobs[data_path])} "
        "per-tile overlay jobs"
    )



In [ ]:
# Per-ROI overview: fan out one slurm job per ROI. Each job renders a
# stitched RGB overview of the reference round with red dots at surviving
# soma centroids (global x/y). ROIs with no survivors are still drawn so
# the output set is uniform.
from iss_preprocess.diagnostics.diag_somata import plot_filtered_soma_overlays

filtered_overlay_jobs = {}
for data_path in qc_overlay_chambers:
    filtered_overlay_jobs[data_path] = plot_filtered_soma_overlays(
        data_path,
        barcode_prefix=barcode_prefix,
        downsample_factor=4,
        use_slurm=True,
    )
    print(
        f"{data_path}: submitted {len(filtered_overlay_jobs[data_path])} "
        "ROI overlay jobs"
    )



## 15. Explore QC Metrics And Spatial Positions

This is the same general threshold-tuning stage as the exploratory notebooks,
but now done from the stitched ROI-global soma table.

In [ ]:
example_data_path = data_paths[1]
for data_path in data_paths:
    all_somata_df = load_stitched_soma_calls(data_path, barcode_prefix=barcode_prefix, filtered=False)
    filtered_df = load_stitched_soma_calls(data_path, barcode_prefix=barcode_prefix, filtered=True)

    plot_qc_histograms(all_somata_df, title=f"{data_path} | all somata")
    plot_qc_histograms(filtered_df, title=f"{data_path} | filtered somata")


## 16. Combine Filtered Somata Across Chambers

This is convenient for later barcode-level diagnostics.

In [ ]:
all_filtered = []

for data_path in data_paths:
    df = load_stitched_soma_calls(data_path, barcode_prefix=barcode_prefix, filtered=True).copy()
    df["data_path"] = data_path
    all_filtered.append(df)

all_filtered = pd.concat(all_filtered, ignore_index=True) if all_filtered else pd.DataFrame()
print(f"Combined filtered somata: {len(all_filtered)} rows")
all_filtered.head()

## 17. Optional: Compare Called Barcodes To The Sindbis Library

Set `sindbis_library_mat` above to enable this section.
The notebook expects the same `.mat` structure used in the scratch notebook.

In [ ]:
if sindbis_library_mat:
    library_sequences = load_sindbis_library_sequences(sindbis_library_mat)
    query_sequences = pd.Index(all_filtered["bases"].dropna().unique())
    min_dists = min_hamming_distance_to_library(query_sequences, library_sequences)

    plt.figure(figsize=(6, 4))
    plt.hist(min_dists, bins=np.arange(-0.5, 5.5, 1))
    plt.xlabel("minimum hamming distance to Sindbis library")
    plt.ylabel("count")
    plt.show()

    pd.DataFrame({
        "bases": query_sequences,
        "min_hamming_to_library": min_dists,
    }).sort_values("min_hamming_to_library").head(20)
else:
    print("Set sindbis_library_mat to a valid .mat file path to enable library diagnostics.")

## 18. Optional: Duplicate-Barcode Diagnostics

This mirrors the duplicate-barcode exploration from the scratch notebook.
It is useful for spotting suspicious repeated soma barcodes within a slice or
across nearby ROIs.

In [ ]:
duplicate_candidates = all_filtered.groupby("bases").filter(lambda x: len(x) > 1).copy()
duplicate_candidates["barcode_count"] = duplicate_candidates["bases"].map(
    duplicate_candidates["bases"].value_counts()
)

print(f"Duplicate-barcode rows: {len(duplicate_candidates)}")
duplicate_candidates[["bases", "barcode_count", "roi", "x", "y", "data_path"]].head()

In [ ]:
dists = []

for (_, roi, data_path), group in duplicate_candidates.groupby(["bases", "roi", "data_path"]):
    pts = group[["x", "y"]].to_numpy()
    if len(pts) < 2:
        continue
    for i, j in combinations(range(len(pts)), 2):
        dists.append(np.linalg.norm(pts[i] - pts[j]))

dists = np.asarray(dists)
print(f"Computed {len(dists)} within-ROI duplicate distances")

if len(dists):
    dists_plot = dists[dists < 10000]
    plt.figure(figsize=(6, 4))
    plt.hist(dists_plot, bins=50)
    plt.xlabel("within-ROI Euclidean distance (pixels)")
    plt.ylabel("count")

    plt.show()

## 19. Outputs Summary

The key outputs per chamber are:
- raw segmentation masks: `processed/cells/<segmentation_prefix>_masks_<roi>_<tilex>_<tiley>.npy`
- soma atlas labels: `processed/cells/soma_atlas_<roi>.npy`
- soma atlas metadata: `processed/cells/soma_atlas_<roi>_meta.npz`
- per-tile soma trace cache: `processed/somata_traces/barcode_round_soma_traces_*.pkl`
- per-tile soma trace stats: `processed/somata_traces/barcode_round_soma_trace_stats_*.json`
- cached trace stats summary: `processed/somata_traces/barcode_round_soma_trace_tile_stats.pkl`
- per-tile soma calls: `processed/cells/barcode_round_somata_<roi>_<tilex>_<tiley>.pkl`
- stitched ROI-global soma calls: `processed/cells/barcode_round_cells/barcode_round_df_corrected.pkl`
- filtered soma calls: `processed/cells/barcode_round_cells/barcode_round_df_filtered.pkl`

In **shared mode**, the reference is mouse-wide:
- shared cluster means: `processed/{project}/{mouse}/somata_barcode_cluster_means.npy`
- shared reference barcodes: `processed/{project}/{mouse}/somata_reference_barcodes.npz`
- shared selected reference traces: `processed/{project}/{mouse}/somata_traces_df.pkl`
- shared diagnostics: `processed/{project}/{mouse}/diagnostics/shared_soma_reference/`

In **per-chamber mode**, the same artifacts live under each chamber's own
`processed/` folder (`somata_barcode_cluster_means.npy`,
`somata_reference_barcodes.npz`, `somata_traces_df.pkl`).


In [ ]:
summary_rows = []
for data_path in data_paths:
    outputs = get_soma_output_paths(data_path, barcode_prefix=barcode_prefix)
    summary_rows.append({"data_path": data_path, **outputs})
pd.DataFrame(summary_rows)

## 20. Frame Conventions

Frame transitions:

1. raw per-tile segmentation masks
   - segmentation-tile coordinates
   - soma segmentation preserves tile-edge labels; no pre-atlas edge trimming

2. `build_soma_atlases(...)` (one-time, per ROI)
   - reads raw masks by default (`mask_suffix=""`)
   - composes one direct mask-to-global barcode-reference affine per tile
   - expands the atlas canvas before warping, storing `canvas_global_origin_yx`
   - assigns stable global integer soma IDs
   - merges tile-border duplicates by containment, arbitrates true different-gid
     pixel conflicts via distance-transform watershed
   - stores `reference_prefix` and `corrected_shifts` in meta so loaders can
     validate basecalling-frame compatibility

3. `load_soma_atlas_tile(...)` (deterministic, on demand)
   - crops the atlas using array-space tile origins derived from the canvas offset
   - returns a per-tile mask in barcode reference-tile coordinates
   - integer labels are global soma IDs (no relabeling)
   - zeroes gids not owned by the requested tile

4. `barcode_round_somata_<roi>_<tilex>_<tiley>.pkl`
   - per-tile soma calls; centroids in barcode reference-tile coords;
     `label` is the global soma ID

5. `stitch_soma_calls(...)`
   - ROI-global coordinates added; dedup by global `label`
